# Phase 6: GEO Tumor-Cohort Support Scoring

Notebook này dùng GEO như một external tumor-only support cohort cho top candidate từ GDC + STRING. GEO không tạo ranking chính, không xác nhận tumor-vs-normal differential expression và không xác nhận protein abundance.

Nguyên tắc chạy:
- GDC + STRING giữ vai trò discovery/ranking chính.
- GEO chỉ chấm điểm hỗ trợ expression tương đối cho candidate gene tương ứng.
- Chỉ đọc `mart/top_candidate_targets` và `refined/geo`; không sửa raw/refined.
- Ghi output vào `analysis/geo_validation_result`, `analysis/geo_validation_summary`, `analysis/geo_supported_top_candidates`.
- Vẽ 3 hình local cho báo cáo: support summary, top supported candidates, GDC score vs GEO support score.

In [1]:
from pathlib import Path

from pyspark.sql import SparkSession, Window, functions as F

# Khai báo đường dẫn HDFS và cấu hình chính cho Phase 6.
HDFS_BASE = "hdfs://master11:9000"
ANALYSIS_PATH = f"{HDFS_BASE}/drugtarget/data/analysis"
MART_PATH = f"{HDFS_BASE}/drugtarget/data/mart"
REFINED_GEO_PATH = f"{HDFS_BASE}/drugtarget/data/refined/geo"

TOP_N = 100
GEO_ALREADY_LOG_TRANSFORMED = True
GEO_VALIDATION_MODE = "tumor_cohort_expression_support"

TOP_TARGETS_PATH = f"{MART_PATH}/top_candidate_targets"
GEO_EXPRESSION_TABLE = "geo.expression"
GEO_METADATA_TABLE = "geo.metadata"
GEO_EXPRESSION_PATH = f"{REFINED_GEO_PATH}/expression/version=v1"
GEO_METADATA_PATH = f"{REFINED_GEO_PATH}/metadata/version=v1"
GEO_VALIDATION_RESULT_PATH = f"{ANALYSIS_PATH}/geo_validation_result"
GEO_VALIDATION_SUMMARY_PATH = f"{ANALYSIS_PATH}/geo_validation_summary"
GEO_SUPPORTED_TOP_CANDIDATES_PATH = f"{ANALYSIS_PATH}/geo_supported_top_candidates"

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "analysis" else Path.cwd()
FIGURE_DIR = PROJECT_ROOT / "output" / "figures"
FIGURE_SUPPORT_SUMMARY = FIGURE_DIR / "geo_support_summary.png"
FIGURE_TOP_SUPPORTED = FIGURE_DIR / "geo_top_supported_candidates.png"
FIGURE_GDC_VS_GEO = FIGURE_DIR / "gdc_vs_geo_support_scatter.png"

spark = (
    SparkSession.builder.appName("drugtarget-gdc-phase6-geo-support-scoring")
    .config("spark.sql.parquet.compression.codec", "snappy")
    .config("spark.sql.sources.partitionOverwriteMode", "dynamic")
    .enableHiveSupport()
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

print(f"Spark master: {spark.sparkContext.master}")
print(f"Input top candidates: {TOP_TARGETS_PATH}")
print(f"Input GEO expression fallback: {GEO_EXPRESSION_PATH}")
print(f"Input GEO metadata fallback: {GEO_METADATA_PATH}")
print(f"Output GEO validation result: {GEO_VALIDATION_RESULT_PATH}")
print(f"Output GEO validation summary: {GEO_VALIDATION_SUMMARY_PATH}")
print(f"Output GEO supported top candidates: {GEO_SUPPORTED_TOP_CANDIDATES_PATH}")

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/06/03 08:19:47 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark master: local[*]
Input top candidates: hdfs://master11:9000/drugtarget/data/mart/top_candidate_targets
Input GEO expression fallback: hdfs://master11:9000/drugtarget/data/refined/geo/expression/version=v1
Input GEO metadata fallback: hdfs://master11:9000/drugtarget/data/refined/geo/metadata/version=v1
Output GEO validation result: hdfs://master11:9000/drugtarget/data/analysis/geo_validation_result
Output GEO validation summary: hdfs://master11:9000/drugtarget/data/analysis/geo_validation_summary
Output GEO supported top candidates: hdfs://master11:9000/drugtarget/data/analysis/geo_supported_top_candidates


In [2]:
def read_table_or_parquet(table_name: str, parquet_path: str):
    """Đọc Hive table nếu có; nếu chưa có thì đọc Parquet trên HDFS."""
    try:
        frame = spark.table(table_name)
        print(f"Đọc Hive table thành công: {table_name}")
        return frame
    except Exception as exc:
        print(f"Không đọc được Hive table {table_name}; chuyển sang Parquet HDFS.")
        print(f"Lý do: {type(exc).__name__}: {exc}")
        print(f"Đọc Parquet: {parquet_path}")
        return spark.read.parquet(parquet_path)


def require_columns(frame, required_columns, frame_name: str) -> None:
    """Dừng pipeline nếu DataFrame thiếu cột bắt buộc."""
    missing_columns = sorted(set(required_columns) - set(frame.columns))
    if missing_columns:
        raise ValueError(f"{frame_name} thiếu cột bắt buộc: {', '.join(missing_columns)}")


top_candidates = spark.read.parquet(TOP_TARGETS_PATH).filter(F.col("rank") <= F.lit(TOP_N)).cache()
geo_expr_wide = read_table_or_parquet(GEO_EXPRESSION_TABLE, GEO_EXPRESSION_PATH)
geo_meta = read_table_or_parquet(GEO_METADATA_TABLE, GEO_METADATA_PATH)

TOP_REQUIRED_COLUMNS = ["rank", "gene_name", "protein_id", "ensp_id", "final_score"]
GEO_EXPR_REQUIRED_COLUMNS = ["gene_name"]
GEO_META_REQUIRED_COLUMNS = ["Array"]
require_columns(top_candidates, TOP_REQUIRED_COLUMNS, "top_candidate_targets")
require_columns(geo_expr_wide, GEO_EXPR_REQUIRED_COLUMNS, "geo.expression")
require_columns(geo_meta, GEO_META_REQUIRED_COLUMNS, "geo.metadata")

top_candidate_count = top_candidates.count()
if top_candidate_count != TOP_N:
    raise AssertionError(f"Phase 6 cần đúng {TOP_N} top candidates, hiện có {top_candidate_count}.")

print("Schema top_candidate_targets:")
top_candidates.printSchema()
print("Schema GEO expression wide:")
geo_expr_wide.printSchema()
print("Schema GEO metadata:")
geo_meta.printSchema()
print(f"Số candidate dùng cho Phase 6: {top_candidate_count}")

26/06/03 08:19:54 WARN HiveConf: HiveConf of name hive.stats.jdbc.timeout does not exist
26/06/03 08:19:54 WARN HiveConf: HiveConf of name hive.stats.retries.wait does not exist


26/06/03 08:19:56 WARN ObjectStore: Version information not found in metastore. hive.metastore.schema.verification is not enabled so recording the schema version 2.3.0
26/06/03 08:19:56 WARN ObjectStore: setMetaStoreSchemaVersion called but recording version is disabled: version = 2.3.0, comment = Set by MetaStore dis@100.82.104.4


26/06/03 08:19:57 WARN ObjectStore: Failed to get database global_temp, returning NoSuchObjectException


26/06/03 08:19:57 WARN ObjectStore: Failed to get database geo, returning NoSuchObjectException


Không đọc được Hive table geo.expression; chuyển sang Parquet HDFS.
Lý do: AnalysisException: [TABLE_OR_VIEW_NOT_FOUND] The table or view `geo`.`expression` cannot be found. Verify the spelling and correctness of the schema and catalog.
If you did not qualify the name with a schema, verify the current_schema() output, or qualify the name with the correct schema and catalog.
To tolerate the error on drop use DROP VIEW IF EXISTS or DROP TABLE IF EXISTS.;
'UnresolvedRelation [geo, expression], [], false

Đọc Parquet: hdfs://master11:9000/drugtarget/data/refined/geo/expression/version=v1


26/06/03 08:19:58 WARN ObjectStore: Failed to get database geo, returning NoSuchObjectException


Không đọc được Hive table geo.metadata; chuyển sang Parquet HDFS.
Lý do: AnalysisException: [TABLE_OR_VIEW_NOT_FOUND] The table or view `geo`.`metadata` cannot be found. Verify the spelling and correctness of the schema and catalog.
If you did not qualify the name with a schema, verify the current_schema() output, or qualify the name with the correct schema and catalog.
To tolerate the error on drop use DROP VIEW IF EXISTS or DROP TABLE IF EXISTS.;
'UnresolvedRelation [geo, metadata], [], false

Đọc Parquet: hdfs://master11:9000/drugtarget/data/refined/geo/metadata/version=v1


Schema top_candidate_targets:
root
 |-- rank: integer (nullable = true)
 |-- gene_name: string (nullable = true)
 |-- gene_id_base: string (nullable = true)
 |-- protein_id: string (nullable = true)
 |-- ensp_id: string (nullable = true)
 |-- protein_name: string (nullable = true)
 |-- log2FC: double (nullable = true)
 |-- p_value: double (nullable = true)
 |-- deg_direction: string (nullable = true)
 |-- degree_protein: long (nullable = true)
 |-- weighted_degree_protein: double (nullable = true)
 |-- num_interactions_in_deg_network: long (nullable = true)
 |-- avg_combined_score: double (nullable = true)
 |-- expression_score: double (nullable = true)
 |-- protein_network_score: double (nullable = true)
 |-- string_confidence_score: double (nullable = true)
 |-- final_score: double (nullable = true)

Schema GEO expression wide:
root
 |-- gene_name: string (nullable = true)
 |-- CL2005060332AA: string (nullable = true)
 |-- CL2004111017AA: string (nullable = true)
 |-- CL20041110103AA

In [3]:
# Xác nhận sample ID expression wide có metadata tương ứng.
sample_columns = [col_name for col_name in geo_expr_wide.columns if col_name != "gene_name"]
if not sample_columns:
    raise ValueError("GEO expression không có cột sample nào ngoài gene_name.")

geo_expr_sample_ids = spark.createDataFrame([(sample_id,) for sample_id in sample_columns], ["sample_id"])
geo_expr_sample_ids = geo_expr_sample_ids.withColumn("sample_id", F.upper(F.trim(F.col("sample_id"))))

geo_meta_clean = geo_meta.withColumn("sample_id", F.upper(F.trim(F.col("Array"))))
geo_meta_sample_ids = geo_meta_clean.select("sample_id").filter(F.col("sample_id").isNotNull()).distinct()
matched_geo_sample_ids = geo_expr_sample_ids.join(geo_meta_sample_ids, on="sample_id", how="inner").cache()

num_expr_samples = geo_expr_sample_ids.count()
num_meta_samples = geo_meta_sample_ids.count()
geo_total_samples = matched_geo_sample_ids.count()

print(f"Expression samples: {num_expr_samples}")
print(f"Metadata samples: {num_meta_samples}")
print(f"Matched GEO tumor-cohort samples: {geo_total_samples}")

if geo_total_samples == 0:
    raise ValueError("Không có sample nào khớp giữa GEO expression và metadata sau khi upper/trim sample ID.")

Expression samples: 1106
Metadata samples: 1106
Matched GEO tumor-cohort samples: 1106


In [4]:
# Chuẩn hóa top candidate và lọc GEO expression theo candidate genes trước khi unpivot.
top_candidates_clean = (
    top_candidates
    .select(
        F.col("rank").cast("int").alias("rank"),
        F.col("gene_name"),
        F.upper(F.trim(F.col("gene_name"))).alias("gene_name_norm"),
        F.col("protein_id"),
        F.col("ensp_id"),
        F.col("final_score").cast("double").alias("final_score"),
    )
    .filter(F.col("gene_name_norm").isNotNull() & (F.col("gene_name_norm") != F.lit("")))
    .cache()
)

candidate_gene_keys = top_candidates_clean.select("gene_name_norm").distinct()

geo_expr_filtered = (
    geo_expr_wide
    .withColumn("gene_name_norm", F.upper(F.trim(F.col("gene_name"))))
    .filter(F.col("gene_name_norm").isNotNull() & (F.col("gene_name_norm") != F.lit("")))
    .join(candidate_gene_keys, on="gene_name_norm", how="inner")
    .cache()
)

num_top_rows = top_candidates_clean.count()
num_top_genes = candidate_gene_keys.count()
num_matched_geo_genes = geo_expr_filtered.select("gene_name_norm").distinct().count()

if num_top_rows != top_candidate_count:
    raise AssertionError(f"Số candidate sau chuẩn hóa không khớp input: {num_top_rows} vs {top_candidate_count}.")

print(f"Top candidate rows: {num_top_rows}")
print(f"Distinct top candidate genes: {num_top_genes}")
print(f"Matched GEO candidate genes: {num_matched_geo_genes}")

26/06/03 08:20:09 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


Top candidate rows: 100
Distinct top candidate genes: 100
Matched GEO candidate genes: 82


In [5]:
# Chuyển expression wide sang long sau khi đã lọc candidate genes.
stack_expression = ", ".join([f"'{sample_id}', `{sample_id}`" for sample_id in sample_columns])

geo_expr_long = (
    geo_expr_filtered
    .selectExpr(
        "gene_name_norm",
        f"stack({len(sample_columns)}, {stack_expression}) as (sample_id, expression_raw)",
    )
    .withColumn("sample_id", F.upper(F.trim(F.col("sample_id"))))
    .join(matched_geo_sample_ids, on="sample_id", how="inner")
    .withColumn("expression_value", F.col("expression_raw").cast("double"))
    .drop("expression_raw")
    .filter(F.col("expression_value").isNotNull())
)

if GEO_ALREADY_LOG_TRANSFORMED:
    geo_expr_long = geo_expr_long.withColumn("geo_expression_used", F.col("expression_value"))
else:
    geo_expr_long = geo_expr_long.withColumn("geo_expression_used", F.log2(F.col("expression_value") + F.lit(1.0)))

# Nếu GEO còn nhiều probe/row cho cùng gene/sample, dùng trung bình để đưa về một giá trị.
geo_expr_gene_sample = (
    geo_expr_long
    .groupBy("gene_name_norm", "sample_id")
    .agg(F.avg("geo_expression_used").alias("geo_expression_used"))
    .cache()
)

print("Khoảng giá trị GEO expression dùng để chấm support:")
geo_expr_gene_sample.agg(
    F.count("*").alias("num_gene_sample_values"),
    F.countDistinct("gene_name_norm").alias("num_genes"),
    F.countDistinct("sample_id").alias("num_samples"),
    F.min("geo_expression_used").alias("min_expression"),
    F.max("geo_expression_used").alias("max_expression"),
    F.avg("geo_expression_used").alias("avg_expression"),
).show(truncate=False)

Khoảng giá trị GEO expression dùng để chấm support:


+----------------------+---------+-----------+--------------+--------------+---------------------+
|num_gene_sample_values|num_genes|num_samples|min_expression|max_expression|avg_expression       |
+----------------------+---------+-----------+--------------+--------------+---------------------+
|85129                 |82       |1106       |-5.920062947  |8.783807556   |-0.019034559807821073|
+----------------------+---------+-----------+--------------+--------------+---------------------+



In [6]:
# Tính percentile expression trong từng sample trên chính tập candidate genes có dữ liệu.
sample_rank_window = Window.partitionBy("sample_id").orderBy(F.asc("geo_expression_used"), F.asc("gene_name_norm"))
sample_count_window = Window.partitionBy("sample_id")

geo_expr_ranked = (
    geo_expr_gene_sample
    .withColumn("candidate_gene_count_in_sample", F.count("*").over(sample_count_window))
    .withColumn("geo_percent_rank", F.percent_rank().over(sample_rank_window))
    .withColumn(
        "geo_expression_percentile",
        F.when(F.col("candidate_gene_count_in_sample") <= F.lit(1), F.lit(1.0))
        .otherwise(F.col("geo_percent_rank")),
    )
    .withColumn("is_top_quartile", (F.col("geo_expression_percentile") >= F.lit(0.75)).cast("int"))
    .cache()
)

geo_gene_stats = (
    geo_expr_ranked
    .groupBy("gene_name_norm")
    .agg(
        F.countDistinct("sample_id").cast("long").alias("geo_num_samples_available"),
        F.avg("geo_expression_used").alias("geo_mean_expression"),
        F.expr("percentile_approx(geo_expression_used, 0.5)").alias("geo_median_expression"),
        F.avg("geo_expression_percentile").alias("geo_mean_percentile"),
        (F.sum("is_top_quartile") / F.lit(float(geo_total_samples))).alias("geo_top_quartile_rate"),
    )
    .withColumn("geo_total_samples", F.lit(geo_total_samples).cast("long"))
    .withColumn("geo_coverage_rate", F.col("geo_num_samples_available") / F.col("geo_total_samples"))
)

OUTPUT_COLUMNS = [
    "rank",
    "gene_name",
    "gene_name_norm",
    "protein_id",
    "ensp_id",
    "final_score",
    "geo_match_status",
    "geo_num_samples_available",
    "geo_total_samples",
    "geo_coverage_rate",
    "geo_mean_expression",
    "geo_median_expression",
    "geo_mean_percentile",
    "geo_top_quartile_rate",
    "geo_support_score",
    "geo_support_level",
    "geo_validation_mode",
]

geo_validation_result = (
    top_candidates_clean
    .join(geo_gene_stats, on="gene_name_norm", how="left")
    .withColumn(
        "geo_match_status",
        F.when(F.col("geo_num_samples_available").isNull(), F.lit("Not Found"))
        .otherwise(F.lit("Found in GEO cohort")),
    )
    .withColumn("geo_total_samples", F.coalesce(F.col("geo_total_samples"), F.lit(geo_total_samples).cast("long")))
    .withColumn(
        "geo_support_score",
        F.when(F.col("geo_match_status") == F.lit("Not Found"), F.lit(None).cast("double"))
        .otherwise(
            F.lit(0.2) * F.col("geo_coverage_rate")
            + F.lit(0.5) * F.col("geo_mean_percentile")
            + F.lit(0.3) * F.col("geo_top_quartile_rate")
        ),
    )
    .withColumn(
        "geo_support_level",
        F.when(F.col("geo_match_status") == F.lit("Not Found"), F.lit("Not Found"))
        .when(F.col("geo_support_score") >= F.lit(0.75), F.lit("Strong GEO support"))
        .when(F.col("geo_support_score") >= F.lit(0.50), F.lit("Moderate GEO support"))
        .otherwise(F.lit("Limited GEO support")),
    )
    .withColumn("geo_validation_mode", F.lit(GEO_VALIDATION_MODE))
    .select(*OUTPUT_COLUMNS)
    .orderBy("rank")
    .cache()
)

level_order_expr = (
    "CASE geo_support_level "
    "WHEN 'Strong GEO support' THEN 1 "
    "WHEN 'Moderate GEO support' THEN 2 "
    "WHEN 'Limited GEO support' THEN 3 "
    "WHEN 'Not Found' THEN 4 ELSE 5 END"
)

geo_validation_summary = (
    geo_validation_result
    .groupBy("geo_support_level")
    .agg(F.count("*").cast("long").alias("candidate_count"))
    .withColumn("candidate_percentage", F.col("candidate_count") / F.lit(float(top_candidate_count)) * F.lit(100.0))
    .orderBy(F.expr(level_order_expr))
    .cache()
)

support_rank_window = Window.orderBy(F.desc_nulls_last("geo_support_score"), F.asc("rank"))
GEO_SUPPORTED_TOP_COLUMNS = [
    "geo_support_rank",
    "rank",
    "gene_name",
    "protein_id",
    "ensp_id",
    "final_score",
    "geo_coverage_rate",
    "geo_mean_percentile",
    "geo_top_quartile_rate",
    "geo_support_score",
    "geo_support_level",
]
geo_supported_top_candidates = (
    geo_validation_result
    .withColumn("geo_support_rank", F.row_number().over(support_rank_window))
    .select(*GEO_SUPPORTED_TOP_COLUMNS)
    .orderBy("geo_support_rank")
    .cache()
)

print("Schema geo_validation_result:")
geo_validation_result.printSchema()
print("Summary Phase 6:")
geo_validation_summary.show(truncate=False)
print("Top GEO-supported candidates:")
geo_supported_top_candidates.show(30, truncate=False)

26/06/03 08:20:50 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/03 08:20:50 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


Schema geo_validation_result:
root
 |-- rank: integer (nullable = true)
 |-- gene_name: string (nullable = true)
 |-- gene_name_norm: string (nullable = true)
 |-- protein_id: string (nullable = true)
 |-- ensp_id: string (nullable = true)
 |-- final_score: double (nullable = true)
 |-- geo_match_status: string (nullable = false)
 |-- geo_num_samples_available: long (nullable = true)
 |-- geo_total_samples: long (nullable = false)
 |-- geo_coverage_rate: double (nullable = true)
 |-- geo_mean_expression: double (nullable = true)
 |-- geo_median_expression: double (nullable = true)
 |-- geo_mean_percentile: double (nullable = true)
 |-- geo_top_quartile_rate: double (nullable = true)
 |-- geo_support_score: double (nullable = true)
 |-- geo_support_level: string (nullable = false)
 |-- geo_validation_mode: string (nullable = false)

Summary Phase 6:


+--------------------+---------------+--------------------+
|geo_support_level   |candidate_count|candidate_percentage|
+--------------------+---------------+--------------------+
|Moderate GEO support|47             |47.0                |
|Limited GEO support |35             |35.0                |
|Not Found           |18             |18.0                |
+--------------------+---------------+--------------------+

Top GEO-supported candidates:


26/06/03 08:21:16 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/03 08:21:16 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


26/06/03 08:21:17 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/03 08:21:17 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


+----------------+----+---------+--------------------+---------------+-------------------+------------------+-------------------+---------------------+------------------+--------------------+
|geo_support_rank|rank|gene_name|protein_id          |ensp_id        |final_score        |geo_coverage_rate |geo_mean_percentile|geo_top_quartile_rate|geo_support_score |geo_support_level   |
+----------------+----+---------+--------------------+---------------+-------------------+------------------+-------------------+---------------------+------------------+--------------------+
|1               |66  |COL1A1   |9606.ENSP00000225964|ENSP00000225964|0.3391067705821269 |1.0               |0.5502681086157938 |0.32640144665461124  |0.5730544883042803|Moderate GEO support|
|2               |50  |CCNB1    |9606.ENSP00000256442|ENSP00000256442|0.34970797480216054|1.0               |0.5404537273558965 |0.337251356238698    |0.5714022705495576|Moderate GEO support|
|3               |10  |SPP1     |9606.EN

In [7]:
# Ghi output Phase 6 vào layer analysis, không ghi raw/refined.
geo_validation_result.write.mode("overwrite").option("compression", "snappy").parquet(GEO_VALIDATION_RESULT_PATH)
geo_validation_summary.write.mode("overwrite").option("compression", "snappy").parquet(GEO_VALIDATION_SUMMARY_PATH)
geo_supported_top_candidates.write.mode("overwrite").option("compression", "snappy").parquet(GEO_SUPPORTED_TOP_CANDIDATES_PATH)

print(f"Đã ghi geo_validation_result: {GEO_VALIDATION_RESULT_PATH}")
print(f"Đã ghi geo_validation_summary: {GEO_VALIDATION_SUMMARY_PATH}")
print(f"Đã ghi geo_supported_top_candidates: {GEO_SUPPORTED_TOP_CANDIDATES_PATH}")

Đã ghi geo_validation_result: hdfs://master11:9000/drugtarget/data/analysis/geo_validation_result
Đã ghi geo_validation_summary: hdfs://master11:9000/drugtarget/data/analysis/geo_validation_summary
Đã ghi geo_supported_top_candidates: hdfs://master11:9000/drugtarget/data/analysis/geo_supported_top_candidates


In [8]:
# Vẽ 3 hình báo cáo bằng Pillow để không phụ thuộc matplotlib.
FIGURE_DIR.mkdir(parents=True, exist_ok=True)


def load_font(size: int, bold: bool = False):
    from PIL import ImageFont

    candidates = [
        "/usr/share/fonts/truetype/liberation/LiberationSans-Bold.ttf" if bold else "/usr/share/fonts/truetype/liberation/LiberationSans-Regular.ttf",
        "/usr/share/fonts/opentype/urw-base35/NimbusSans-Bold.otf" if bold else "/usr/share/fonts/opentype/urw-base35/NimbusSans-Regular.otf",
    ]
    for candidate in candidates:
        try:
            return ImageFont.truetype(candidate, size=size)
        except Exception:
            pass
    return ImageFont.load_default()


def text_size(draw, text: str, font):
    box = draw.textbbox((0, 0), text, font=font)
    return box[2] - box[0], box[3] - box[1]


def support_color(level: str) -> str:
    return {
        "Strong GEO support": "#2ca66f",
        "Moderate GEO support": "#c98721",
        "Limited GEO support": "#c94b5b",
        "Not Found": "#8a9892",
    }.get(level, "#8a9892")


def draw_summary(path, rows):
    from PIL import Image, ImageDraw

    width, height = 1100, 680
    img = Image.new("RGB", (width, height), "white")
    draw = ImageDraw.Draw(img)
    title_font = load_font(30, True)
    axis_font = load_font(19)
    label_font = load_font(17)
    value_font = load_font(22, True)
    title = "GEO support summary"
    subtitle = "Top 100 GDC + STRING candidates | tumor-only GEO expression support"
    tw, _ = text_size(draw, title, title_font)
    sw, _ = text_size(draw, subtitle, axis_font)
    draw.text(((width - tw) / 2, 26), title, fill="#17201d", font=title_font)
    draw.text(((width - sw) / 2, 66), subtitle, fill="#62716b", font=axis_font)
    left, right, top, bottom = 120, 70, 132, 150
    baseline = height - bottom
    plot_w = width - left - right
    plot_h = height - top - bottom
    counts = [int(row["candidate_count"]) for row in rows]
    max_count = max(counts) if counts else 1
    y_max = max(10, ((max_count + 9) // 10) * 10)
    for tick in range(0, y_max + 1, max(1, y_max // 5)):
        y = baseline - (tick / y_max) * plot_h
        draw.line((left, y, width - right, y), fill="#dce5e0")
        label = str(tick)
        lw, lh = text_size(draw, label, label_font)
        draw.text((left - lw - 12, y - lh / 2), label, fill="#17201d", font=label_font)
    slot = plot_w / max(len(rows), 1)
    for index, row in enumerate(rows):
        count = int(row["candidate_count"])
        level = row["geo_support_level"]
        bar_w = min(185, slot * 0.55)
        x0 = left + index * slot + (slot - bar_w) / 2
        x1 = x0 + bar_w
        y0 = baseline - (count / y_max) * plot_h
        draw.rectangle((x0, y0, x1, baseline), fill=support_color(level), outline="#17201d", width=2)
        value = str(count)
        vw, vh = text_size(draw, value, value_font)
        draw.text(((x0 + x1 - vw) / 2, y0 - vh - 9), value, fill="#17201d", font=value_font)
        label_lines = level.replace(" GEO ", "\nGEO ").split("\n")
        line_y = baseline + 18
        for line in label_lines:
            lw, lh = text_size(draw, line, label_font)
            draw.text(((x0 + x1 - lw) / 2, line_y), line, fill="#17201d", font=label_font)
            line_y += lh + 4
    img.save(path, format="PNG")


def draw_top_supported(path, rows):
    from PIL import Image, ImageDraw

    width, height = 1200, 820
    img = Image.new("RGB", (width, height), "white")
    draw = ImageDraw.Draw(img)
    title_font = load_font(29, True)
    label_font = load_font(16)
    value_font = load_font(17, True)
    draw.text((44, 28), "Top GEO-supported candidates", fill="#17201d", font=title_font)
    draw.text((44, 66), "Bar length = geo_support_score; ranking from GEO support only", fill="#62716b", font=label_font)
    left, right, top, bottom = 170, 80, 110, 42
    plot_w = width - left - right
    row_h = (height - top - bottom) / max(len(rows), 1)
    for index, row in enumerate(rows):
        y = top + index * row_h + 5
        score = float(row["geo_support_score"] or 0)
        bar_w = max(1, min(1, score) * plot_w)
        level = row["geo_support_level"]
        draw.text((30, y + row_h * 0.62 - 10), str(row["gene_name"]), fill="#17201d", font=value_font)
        draw.rectangle((left, y, left + bar_w, y + row_h * 0.55), fill=support_color(level))
        draw.rectangle((left, y, left + plot_w, y + row_h * 0.55), outline="#dce5e0")
        draw.text((left + bar_w + 8, y + row_h * 0.62 - 10), f"{score:.3f} | rank {row['rank']}", fill="#34433e", font=label_font)
    img.save(path, format="PNG")


def draw_scatter(path, rows):
    from PIL import Image, ImageDraw

    width, height = 1100, 760
    img = Image.new("RGB", (width, height), "white")
    draw = ImageDraw.Draw(img)
    title_font = load_font(29, True)
    label_font = load_font(17)
    small_font = load_font(14)
    draw.text((44, 28), "GDC score vs GEO support score", fill="#17201d", font=title_font)
    draw.text((44, 66), "X = final_score from GDC + STRING; Y = GEO support score", fill="#62716b", font=label_font)
    left, right, top, bottom = 82, 210, 112, 74
    plot_w = width - left - right
    plot_h = height - top - bottom
    xs = [float(row["final_score"] or 0) for row in rows]
    ys = [float(row["geo_support_score"] or 0) for row in rows]
    max_x = max(xs) if xs else 1.0
    max_y = max(ys) if ys else 1.0
    x_max = max(0.1, max_x * 1.08)
    y_max = max(0.1, max_y * 1.08)
    draw.rectangle((left, top, left + plot_w, top + plot_h), outline="#dce5e0", width=2)
    for tick in [0, 0.25, 0.5, 0.75, 1.0]:
        x = left + tick * plot_w
        y = top + plot_h - tick * plot_h
        draw.line((x, top, x, top + plot_h), fill="#eef3f0")
        draw.line((left, y, left + plot_w, y), fill="#eef3f0")
        draw.text((x - 12, top + plot_h + 10), f"{tick * x_max:.2f}", fill="#62716b", font=small_font)
        draw.text((18, y - 8), f"{tick * y_max:.2f}", fill="#62716b", font=small_font)
    for row in rows:
        x = left + (float(row["final_score"] or 0) / x_max) * plot_w
        y = top + plot_h - (float(row["geo_support_score"] or 0) / y_max) * plot_h
        color = support_color(row["geo_support_level"])
        draw.ellipse((x - 5, y - 5, x + 5, y + 5), fill=color, outline="#17201d")
        if int(row["rank"]) <= 10:
            draw.text((x + 7, y - 7), str(row["gene_name"]), fill="#17201d", font=small_font)
    draw.text((left + plot_w / 2 - 70, height - 34), "final_score", fill="#17201d", font=label_font)
    draw.text((20, top - 26), "geo_support_score", fill="#17201d", font=label_font)
    legend_y = top
    for level in ["Strong GEO support", "Moderate GEO support", "Limited GEO support", "Not Found"]:
        draw.rectangle((left + plot_w + 42, legend_y, left + plot_w + 62, legend_y + 20), fill=support_color(level))
        draw.text((left + plot_w + 70, legend_y), level, fill="#17201d", font=small_font)
        legend_y += 30
    img.save(path, format="PNG")


summary_rows = [row.asDict() for row in geo_validation_summary.collect()]
top_supported_rows = [row.asDict() for row in geo_supported_top_candidates.limit(20).collect()]
scatter_rows = [row.asDict() for row in geo_validation_result.collect()]

draw_summary(FIGURE_SUPPORT_SUMMARY, summary_rows)
draw_top_supported(FIGURE_TOP_SUPPORTED, top_supported_rows)
draw_scatter(FIGURE_GDC_VS_GEO, scatter_rows)

for figure_path in [FIGURE_SUPPORT_SUMMARY, FIGURE_TOP_SUPPORTED, FIGURE_GDC_VS_GEO]:
    if not figure_path.exists() or figure_path.stat().st_size == 0:
        raise AssertionError(f"Hình GEO chưa tồn tại hoặc rỗng: {figure_path}")
    print(f"Đã ghi hình: {figure_path} ({figure_path.stat().st_size} bytes)")

Đã ghi hình: /home/dis/Drug-Target-Project/output/figures/geo_support_summary.png (26178 bytes)
Đã ghi hình: /home/dis/Drug-Target-Project/output/figures/geo_top_supported_candidates.png (74791 bytes)
Đã ghi hình: /home/dis/Drug-Target-Project/output/figures/gdc_vs_geo_support_scatter.png (42968 bytes)


In [9]:
# Đọc lại output để kiểm tra schema, count, score và thứ tự ranking.
hadoop_conf = spark.sparkContext._jsc.hadoopConfiguration()
for output_path in [GEO_VALIDATION_RESULT_PATH, GEO_VALIDATION_SUMMARY_PATH, GEO_SUPPORTED_TOP_CANDIDATES_PATH]:
    hdfs_path = spark._jvm.org.apache.hadoop.fs.Path(output_path)
    fs = hdfs_path.getFileSystem(hadoop_conf)
    success_path = spark._jvm.org.apache.hadoop.fs.Path(output_path + "/_SUCCESS")
    if not fs.exists(hdfs_path):
        raise AssertionError(f"Output Phase 6 chưa tồn tại trên HDFS: {output_path}")
    if not fs.exists(success_path):
        raise AssertionError(f"Output Phase 6 thiếu _SUCCESS: {output_path}")

result_output = spark.read.parquet(GEO_VALIDATION_RESULT_PATH)
summary_output = spark.read.parquet(GEO_VALIDATION_SUMMARY_PATH)
supported_output = spark.read.parquet(GEO_SUPPORTED_TOP_CANDIDATES_PATH)

if result_output.columns != OUTPUT_COLUMNS:
    raise AssertionError(f"Schema geo_validation_result không đúng: {result_output.columns}")
if summary_output.columns != ["geo_support_level", "candidate_count", "candidate_percentage"]:
    raise AssertionError(f"Schema geo_validation_summary không đúng: {summary_output.columns}")
if supported_output.columns != GEO_SUPPORTED_TOP_COLUMNS:
    raise AssertionError(f"Schema geo_supported_top_candidates không đúng: {supported_output.columns}")

if result_output.count() != TOP_N:
    raise AssertionError(f"geo_validation_result phải có đúng {TOP_N} dòng.")
if supported_output.count() != TOP_N:
    raise AssertionError(f"geo_supported_top_candidates phải có đúng {TOP_N} dòng.")

summary_total = summary_output.agg(F.sum("candidate_count").alias("total_count")).first()["total_count"]
summary_pct = summary_output.agg(F.sum("candidate_percentage").alias("total_percentage")).first()["total_percentage"]
if summary_total != TOP_N:
    raise AssertionError(f"Tổng summary không khớp {TOP_N}: {summary_total}")
if abs(float(summary_pct or 0) - 100.0) > 0.001:
    raise AssertionError(f"Tổng percentage summary không bằng 100: {summary_pct}")

bad_score_count = result_output.filter(
    F.col("geo_support_score").isNotNull()
    & ((F.col("geo_support_score") < F.lit(0.0)) | (F.col("geo_support_score") > F.lit(1.0)))
).count()
if bad_score_count != 0:
    raise AssertionError(f"Có {bad_score_count} dòng geo_support_score ngoài khoảng [0,1].")

bad_not_found_count = result_output.filter(
    (F.col("geo_support_level") == F.lit("Not Found"))
    & F.col("geo_support_score").isNotNull()
).count()
if bad_not_found_count != 0:
    raise AssertionError(f"Có {bad_not_found_count} dòng Not Found nhưng geo_support_score không null.")

bad_found_count = result_output.filter(
    (F.col("geo_support_level") != F.lit("Not Found"))
    & F.col("geo_support_score").isNull()
).count()
if bad_found_count != 0:
    raise AssertionError(f"Có {bad_found_count} dòng Found nhưng thiếu geo_support_score.")

support_order_window = Window.orderBy("geo_support_rank")
bad_support_order_count = (
    supported_output
    .withColumn("prev_score", F.lag("geo_support_score").over(support_order_window))
    .withColumn("prev_rank", F.lag("rank").over(support_order_window))
    .filter(
        F.col("prev_score").isNotNull()
        & F.col("geo_support_score").isNotNull()
        & (
            (F.col("geo_support_score") > F.col("prev_score"))
            | ((F.col("geo_support_score") == F.col("prev_score")) & (F.col("rank") < F.col("prev_rank")))
        )
    )
    .count()
)
if bad_support_order_count != 0:
    raise AssertionError("geo_supported_top_candidates không sort đúng theo geo_support_score desc, rank asc.")

print("Kiểm tra output Phase 6 hoàn tất.")
summary_output.show(truncate=False)

26/06/03 08:21:39 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/03 08:21:39 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/03 08:21:39 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/03 08:21:39 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


Kiểm tra output Phase 6 hoàn tất.
+--------------------+---------------+--------------------+
|geo_support_level   |candidate_count|candidate_percentage|
+--------------------+---------------+--------------------+
|Moderate GEO support|47             |47.0                |
|Limited GEO support |35             |35.0                |
|Not Found           |18             |18.0                |
+--------------------+---------------+--------------------+



In [10]:
# Giải phóng cache sau khi hoàn tất Phase 6.
top_candidates.unpersist()
matched_geo_sample_ids.unpersist()
top_candidates_clean.unpersist()
geo_expr_filtered.unpersist()
geo_expr_gene_sample.unpersist()
geo_expr_ranked.unpersist()
geo_validation_result.unpersist()
geo_validation_summary.unpersist()
geo_supported_top_candidates.unpersist()
print("Hoàn tất Phase 6: GEO tumor-cohort support scoring")

Hoàn tất Phase 6: GEO tumor-cohort support scoring
